In [ ]:
import torch
from torch import nnni

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head scaled dot-product attention.

    ``hidden_size`` is the feature size of each head and of the final output.
    """

    def __init__(
        self,
        query_size: int,
        key_size: int,
        value_size: int,
        hidden_size: int,
        head_count: int,
    ) -> None:
        super().__init__()
        if head_count <= 0:
            raise ValueError("head_count must be greater than 0")
        if hidden_size % head_count != 0:
            raise ValueError(
                f"hidden_size ({hidden_size}) must be divisible by head_count ({head_count})"
            )
        self.d_head = hidden_size // head_count
        self.wq = nn.Linear(query_size, hidden_size)
        self.wk = nn.Linear(key_size, hidden_size)
        self.wv = nn.Linear(value_size, hidden_size)
        self.wo = nn.Linear(hidden_size, hidden_size)
        self.head_count = head_count
        self.attention = DotProductAttention()
        self.attention_weights: torch.Tensor | None = None

    def forward(
        self,
        queries: torch.Tensor,
        keys: torch.Tensor,
        values: torch.Tensor,
        masked_position: torch.Tensor,
    ) -> torch.Tensor:
        projected_queries = self.split_heads_for_attention(self.wq(queries))
        projected_keys = self.split_heads_for_attention(self.wk(keys))
        projected_values = self.split_heads_for_attention(self.wv(values))

        output = self.attention(
            projected_queries,
            projected_keys,
            projected_values,
        )
        batch_size = queries.shape[0]
        self.attention_weights = self.attention.attention_weights.reshape(
            batch_size,
            self.head_count,
            queries.shape[1],
            keys.shape[1],
        )
        return self.wo(self.combine_attention(output))

    def split_heads_for_attention(self, tensor: torch.Tensor) -> torch.Tensor:
        tensor = tensor.reshape(
            tensor.shape[0], tensor.shape[1], self.head_count, self.d_head
        )
        tensor = tensor.permute(0, 2, 1, 3)
        return tensor.reshape(-1, tensor.shape[2], tensor.shape[3])

    def combine_attention(self, tensor: torch.Tensor) -> torch.Tensor:
        tensor = tensor.reshape(
            tensor.shape[0] // self.head_count,
            self.head_count,
            tensor.shape[1],
            tensor.shape[2],
        )
        tensor = tensor.permute(0, 2, 1, 3)
        return tensor.reshape(tensor.shape[0], tensor.shape[1], -1)